# Assignment 3: Apache Spark – Products Sold in Each Category

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jits77111-source/DWBDA-Assignment-3-PySpark-Sales-Category/blob/main/Assignment_3_PySpark_Sales_Category.ipynb)

**Author:** Jit Sarkar  
**Course:** Data Warehousing, Big Data & Analytics (DWBDA)  
**GitHub Repository:** [DWBDA-Assignment-3-PySpark-Sales-Category](https://github.com/jits77111-source/DWBDA-Assignment-3-PySpark-Sales-Category)

---

### Objective
Utilize **Apache Spark (PySpark DataFrame API & Spark SQL)** for distributed batch processing to calculate the number of products sold across product categories along with total revenue and category insights.

## 1. Install PySpark & Initialize Spark Session

In [ ]:
# Install PySpark
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg as spark_avg, round as spark_round, count as spark_count
import matplotlib.pyplot as plt

spark = SparkSession.builder \
    .appName("PySparkCategorySales") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark Session Running! Spark Version: {spark.version}")

## 2. Load Sales Dataset into PySpark DataFrame

In [ ]:
import os, urllib.request

dataset_file = "retail_store_sales.csv"
if not os.path.exists(dataset_file):
    url = "https://raw.githubusercontent.com/jits77111-source/DWBDA-Assignment-3-PySpark-Sales-Category/main/retail_store_sales.csv"
    urllib.request.urlretrieve(url, dataset_file)
    print(f"Downloaded '{dataset_file}' from GitHub.")

df_sales = spark.read.csv(dataset_file, header=True, inferSchema=True)
df_sales.show(10, truncate=False)
df_sales.printSchema()

## 3. Calculate Products Sold in Each Category

We group transactions by `Category` and perform `sum("Quantity")` ordered in descending order.

In [ ]:
# Core Assignment Objective
category_sales = df_sales.groupBy("Category") \
    .agg(spark_sum("Quantity").alias("Total_Products_Sold")) \
    .orderBy(col("Total_Products_Sold").desc())

category_sales.show(truncate=False)

## 4. Extended Analysis: Total Sales Revenue by Category

In [ ]:
# Add Revenue Column
df_enriched = df_sales.withColumn(
    "Sales_Amount",
    spark_round(col("Quantity") * col("Unit_Price"), 2)
)

category_summary = df_enriched.groupBy("Category") \
    .agg(
        spark_sum("Quantity").alias("Total_Products_Sold"),
        spark_round(spark_sum("Sales_Amount"), 2).alias("Total_Revenue_USD"),
        spark_round(spark_avg("Unit_Price"), 2).alias("Avg_Unit_Price_USD"),
        spark_count("Transaction_ID").alias("Transaction_Count")
    ) \
    .orderBy(col("Total_Products_Sold").desc())

category_summary.show(truncate=False)

## 5. Spark SQL Querying

In [ ]:
df_enriched.createOrReplaceTempView("sales")

query = """
    SELECT 
        Category,
        SUM(Quantity) AS Total_Units,
        ROUND(SUM(Sales_Amount), 2) AS Total_Sales,
        ROUND(AVG(Sales_Amount), 2) AS Avg_Order_Value
    FROM sales
    GROUP BY Category
    ORDER BY Total_Sales DESC
"""
spark.sql(query).show(truncate=False)

## 6. Visualization

In [ ]:
pdf = category_summary.toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar Chart of Products Sold
ax1.bar(pdf["Category"], pdf["Total_Products_Sold"], color="#2ca02c", edgecolor="black")
ax1.set_title("Total Products Sold by Category", fontsize=14, fontweight="bold")
ax1.set_xlabel("Product Category", fontsize=12)
ax1.set_ylabel("Total Units Sold", fontsize=12)
ax1.tick_params(axis="x", rotation=25)
ax1.grid(True, axis="y", linestyle="--", alpha=0.5)

# Pie Chart of Revenue Share
ax2.pie(
    pdf["Total_Revenue_USD"],
    labels=pdf["Category"],
    autopct="%1.1f%%",
    startangle=140,
    wedgeprops={"edgecolor": "black"}
)
ax2.set_title("Revenue Distribution by Category", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Stop Spark Session
spark.stop()
print("Spark Session cleanly stopped!")